# 02 — Vérification du raccourci de PhiUSIIL

**Objectif** : mesurer sur les 235 795 URL ce que `01` laissait voir sur 10 : les légitimes ont-elles toutes la même forme ? Si oui, une simple règle (sans ML) suffit à résoudre le jeu.

**Méthode** : pour chaque critère, `groupby("label").mean()` donne directement le pourcentage par classe (`0` = phishing, `1` = légitime, confirmé en `01`).

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/raw/PhiUSIIL_Phishing_URL_Dataset.csv")

## 1. HTTPS par classe

In [4]:
# 0 = phishing, 1 = légitime (confirmé dans 01_first_look)
df.groupby("label")["IsHTTPS"].mean()

label
0    0.492238
1    1.000000
Name: IsHTTPS, dtype: float64

**Résultat** : 49,2 % du phishing est en HTTPS, contre **100 %** des légitimes.

## 2. `www.` par classe

`"://www."` plutôt que `"www"` : on ne veut que les noms d'hôte qui **commencent** par `www.` ; `regex=False` pour chercher le point littéralement.

In [5]:
df["has_www"] = df["URL"].str.contains("://www.", regex=False)
df.groupby("label")["has_www"].mean()


label
0    0.413701
1    1.000000
Name: has_www, dtype: float64

**Résultat** : 41,4 % du phishing contre **100 %** des légitimes.

## 3. Les deux critères combinés : l'URL commence-t-elle par `https://www.` ?

In [6]:
df["https_www"] = df["URL"].str.startswith("https://www.")
df.groupby("label")["https_www"].mean()

label
0    0.024261
1    1.000000
Name: https_www, dtype: float64

**Résultat** : 2,4 % du phishing contre **100 %** des légitimes : « ne commence pas par `https://www.` → phishing » attrape déjà 97,6 % du phishing sans aucun faux positif.

## 4. Forme exacte des légitimes : la ligne de base regex

`https://www\.[^/?#]+` avec `fullmatch` : l'URL entière doit être `https://www.` + un domaine, **rien après** (ni chemin, ni paramètres, ni `#`).

In [7]:
pattern = r"https://www\.[^/?#]+"
df["legit_shape"] = df["URL"].str.fullmatch(pattern)
df.groupby("label")["legit_shape"].mean()


label
0    0.006211
1    1.000000
Name: legit_shape, dtype: float64

**Résultat** : 0,62 % du phishing contre **100 %** des légitimes.

## 5. Métriques officielles de la ligne de base

Règle : « l'URL n'a pas la forme légitime → phishing ». Mêmes métriques que pour les futurs modèles (matrice de confusion, précision, rappel, F1).

In [8]:
from sklearn.metrics import confusion_matrix, classification_report

# Prédiction de la ligne de base : 0 (phishing) si l'URL n'a pas la forme légitime
y_true = df["label"]
y_pred = (~df["legit_shape"]).map({True: 0, False: 1})

print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
print(classification_report(y_true, y_pred, labels=[0, 1],
                            target_names=["phishing", "légitime"], digits=4))


[[100318    627]
 [     0 134850]]
              precision    recall  f1-score   support

    phishing     1.0000    0.9938    0.9969    100945
    légitime     0.9954    1.0000    0.9977    134850

    accuracy                         0.9973    235795
   macro avg     0.9977    0.9969    0.9973    235795
weighted avg     0.9974    0.9973    0.9973    235795



**Résultat** : précision phishing **1,0000**, rappel **0,9938**, F1 **0,9969**, accuracy **0,9973**, 0 faux positif — avec une regex d'une ligne.

## 6. Fuite de l'étiquette : `URLSimilarityIndex`

Dans `df.head()`, toutes les légitimes valaient 100. Une colonne qui « contient » déjà la réponse doit être retirée de tout entraînement.

In [9]:
df["sim_100"] = df["URLSimilarityIndex"] == 100
df.groupby("label")["sim_100"].mean()


label
0    0.007786
1    1.000000
Name: sim_100, dtype: float64

**Résultat** : 0,78 % du phishing contre **100 %** des légitimes. `URLSimilarityIndex` fuit l'étiquette → **exclue**.

## Conclusions : vérification du raccourci (PhiUSIIL)

| Critère | Phishing (0) | Légitime (1) |
| --- | --- | --- |
| HTTPS | 49,2 % | 100 % |
| `www.` | 41,4 % | 100 % |
| Commence par `https://www.` | 2,4 % | 100 % |
| Forme exacte `https://www.domaine` | 0,62 % | 100 % |
| `URLSimilarityIndex == 100` | 0,78 % | 100 % |

**Ligne de base regex** (`https://www\.[^/?#]+`, sans ML) :
précision phishing 1,0000, rappel 0,9938, F1 0,9969, accuracy 0,9973.

**Conclusions :**
- Toutes les URL légitimes ont la même forme : le dataset se résout par une regex.
- `URLSimilarityIndex` fuit l'étiquette → à exclure.
- Un score de 97–99,9 % sur PhiUSIIL ne prouve pas la détection du phishing.
- PhiUSIIL = étude de cas du raccourci ; entraînement principal sur Hannousse 2021.
